## Typicality rating experiment — GPT-5.4

In [24]:
import os
# Must be set BEFORE importing transformers, or it will still probe TensorFlow
#os.environ["USE_TF"] = "0"
#os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import re
import pandas as pd
import torch
from transformers import pipeline

#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/"  # change to "meta-llama/Meta-Llama-3-8B-Instruct" if you need 3.0

In [25]:
# Set the language to run this notebook for.
# Must match the suffix used in your CSV's instance_<LANGUAGE> column,
# e.g. "English", "German", "Spanish"
LANGUAGE = "English"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"


In [26]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

4659

In [27]:
df = pd.read_csv('combined_prototypes.csv')
df.head(5)
df = df[df[instance_col].notna() & (df[instance_col].astype(str).str.strip() != "")].reset_index(drop=True)

In [28]:
len(df)

2355

In [29]:
def typicality_prompt(obj, category):
    return f"""
You are participating in a psychology experiment.

Rate how prototypical "{obj}" is as an example of the category "{category}".
Respond with ONLY one integer between 1 and 7, where 1 is not prototypical at all and 7 is very prototypical.
"""

In [30]:
from openai import OpenAI

client = OpenAI(api_key= ['YOUR-API-KEY'], base_url="https://eu.api.openai.com/v1")  # picks up OPENAI_API_KEY from env

In [31]:
def get_typicality(obj, category):
    prompt = typicality_prompt(obj, category)

    response = client.chat.completions.create(
        model="gpt-5.4",  # verify this is your correct/available model string
        messages=[{"role": "user", "content": prompt}],
        #max_tokens=5,
        temperature=0,
    )

    content = response.choices[0].message.content
    match = re.search(r"[1-7]", content)
    if match:
        return int(match.group())
    else:
        print(f"Could not parse response for '{obj}' ({category}): {content!r}")
        return None

In [32]:
# Reset column in case a previous buggy run stored prompt text instead of scores
df["gpt_typicality"] = pd.to_numeric(df["gpt_typicality"], errors="coerce") if "gpt_typicality" in df.columns else None

output_path = f"combined_prototypes_{LANGUAGE}.csv"

for idx, row in df.iterrows():
    if pd.notna(df.at[idx, "gpt_typicality"]):
        continue  # already done, skip (useful on resume)
    score = get_typicality(row[instance_col], row["category"])
    df.at[idx, "gpt_typicality"] = score
    df.to_csv(output_path, index=False)

In [33]:
df.to_csv(f"gpt_{LANGUAGE.lower()}_likert.csv", index=False)


In [34]:
df

,category,concept_en,instance_English,instance_German,instance_Spanish,norm_rating_English,norm_rating_German,norm_rating_Spanish,n_languages,in_multiple_languages,gpt_typicality
0,animal,ANT,ant,Ameise,Hormiga,0.308662,0.630556,0.521127,3,True,5
1,animal,BEE,bee,Biene,Abeja,0.309422,0.711111,0.521127,3,True,5
2,animal,BUTTERFLY,butterfly,Schmetterling,Mariposa,0.298172,0.746667,0.507042,3,True,5
3,animal,CROW,crow,Krähe,Cuervo,0.499310,0.820556,0.394366,3,True,6
4,animal,DUCK,duck,Ente,Pato,0.772608,0.936667,0.830986,3,True,6
...,...,...,...,...,...,...,...,...,...,...,...
2350,women's clothing,STILETTO,stiletto,NaN,NaN,0.364674,NaN,NaN,1,False,5
2351,women's clothing,STOCKINGS,stockings,NaN,NaN,0.492871,NaN,NaN,1,False,5
2352,women's clothing,TIARA,tiara,NaN,NaN,0.124074,NaN,NaN,1,False,2
2353,women's clothing,TUBE TOP,tube top,NaN,NaN,0.477208,NaN,NaN,1,False,5
